# Regime-Aware Equity Behavior Analyzer

## Research Question

How do return, volatility, drawdown, correlation, and trading volume change across different market regimes, and how differently do individual stocks behave when market conditions deteriorate?

## Assets

- AAPL
- MSFT
- JPM
- XOM

## Project Goal

This project studies how equity behavior changes across different market conditions using real historical data.

The analysis will focus on:
- returns
- rolling volatility
- drawdowns
- correlations
- trading volume
- regime comparisons

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

In [2]:
project_root = Path.cwd().parent
data_folder = project_root / "data"
results_folder = project_root / "results"

print("Project root:",
      project_root)

print("\nData folder:",
      data_folder)

Project root: c:\Users\hobat\OneDrive\Desktop\Py file\regime-aware-equity-analyzer

Data folder: c:\Users\hobat\OneDrive\Desktop\Py file\regime-aware-equity-analyzer\data


In [3]:
print("Files in data filder:")

for file in data_folder.iterdir():
    print(file.name)

Files in data filder:
AAPL.csv
JPM.csv
MSFT.csv
XOM.csv


## Stage 2 — Load and Align Multiple Stocks

We will:
- load AAPL, MSFT, JPM, and XOM
- convert the Date column to datetime
- set Date as the index
- keep the Close and Volume columns
- align all stocks on common trading dates

In [4]:
tickers = ["AAPL", "MSFT", "JPM", "XOM"]

stock_data = {}

for ticker in tickers:
    df = pd.read_csv(data_folder / f"{ticker}.csv")

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.set_index("Date").sort_index()

    stock_data[ticker] = df

    print(ticker, df.shape)

AAPL (10567, 5)
MSFT (10184, 5)
JPM (14278, 5)
XOM (14278, 5)


In [5]:
print(stock_data["AAPL"].head())

                Open      High       Low     Close     Volume
Date                                                         
1984-09-07  0.099172  0.100390  0.097975  0.099172   99242379
1984-09-10  0.099172  0.099477  0.096788  0.098584   77028276
1984-09-11  0.099477  0.102175  0.099477  0.100390  181637249
1984-09-12  0.100390  0.100977  0.097367  0.097367  158675628
1984-09-13  0.102784  0.103077  0.102784  0.102784  247131424


In [7]:
close_prices = pd.concat(
    [
        stock_data["AAPL"]["Close"],
        stock_data["MSFT"]["Close"],
        stock_data["JPM"]["Close"],
        stock_data["XOM"]["Close"]
    ],
    axis=1,
    join="inner"
)

close_prices.columns = ["AAPL", "MSFT", "JPM", "XOM"]

print(close_prices.head())
print("\nshape:", close_prices.shape)

                AAPL      MSFT      JPM      XOM
Date                                            
1986-03-13  0.092284  0.069028  6.07788  4.03279
1986-03-14  0.097367  0.069028  6.22230  4.09484
1986-03-17  0.097073  0.069028  6.12120  4.15084
1986-03-18  0.100390  0.069028  6.07788  4.13238
1986-03-19  0.099172  0.069028  6.15009  4.11380

shape: (10182, 4)


In [8]:
volume_data = pd.concat(
    [
        stock_data["AAPL"]["Volume"],
        stock_data["MSFT"]["Volume"],
        stock_data["JPM"]["Volume"],
        stock_data["XOM"]["Volume"]
    ],
    axis=1,
    join="inner"
)

volume_data.columns = ["AAPL", "MSFT", "JPM", "XOM"]

print(volume_data.head())
print("\nShape:", volume_data.shape)

                 AAPL          MSFT           JPM           XOM
Date                                                           
1986-03-13  138223764  1.496329e+09  1.162157e+06  1.797470e+07
1986-03-14  458725551  4.469023e+08  1.160078e+06  2.063858e+07
1986-03-17  141507822  1.931286e+08  7.235918e+05  1.474184e+07
1986-03-18  297219767  9.827675e+07  5.650572e+05  1.750917e+07
1986-03-19  226332410  6.945781e+07  3.191425e+05  1.067231e+07

Shape: (10182, 4)


In [9]:
print("Close missing values:",
      close_prices.isna().sum())

print("\nVolume missing values:",
      volume_data.isna().sum())

print("\nClose duplicate rows:",
      close_prices.duplicated().sum())

print("\nDates sorted?",
      close_prices.index.is_monotonic_increasing)

print("\nShared date range:",
      close_prices.index.min(), "to", close_prices.index.max())

Close missing values: AAPL    0
MSFT    0
JPM     0
XOM     0
dtype: int64

Volume missing values: AAPL    0
MSFT    0
JPM     0
XOM     0
dtype: int64

Close duplicate rows: 2

Dates sorted? True

Shared date range: 1986-03-13 00:00:00 to 2026-08-18 00:00:00


In [10]:
print("Duplicate Close dates:",
      close_prices.index.duplicated().sum())

print("\nDuplicated Volume dates:",
      volume_data.index.duplicated().sum())

Duplicate Close dates: 0

Duplicated Volume dates: 0


In [11]:
duplicated_price_rows = close_prices[
    close_prices.duplicated(keep=False)
]

print(duplicated_price_rows)

                AAPL      MSFT      JPM      XOM
Date                                            
1988-06-06  0.329592  0.291999  4.31164  6.59077
1988-06-07  0.329592  0.291999  4.31164  6.59077
1990-04-24  0.290284  0.553274  3.90776  6.74637
1990-04-25  0.290284  0.553274  3.90776  6.74637


## Stage 3 — Multi-Stock Returns

We convert aligned closing prices into daily returns.

Daily returns let us compare:
- average performance
- volatility
- correlation
- drawdowns
- behavior across market regimes

In [12]:
daily_returns = close_prices.pct_change().dropna()

print(daily_returns.head())
print("\nShape:", daily_returns.shape)

                AAPL  MSFT       JPM       XOM
Date                                          
1986-03-14  0.055077   0.0  0.023762  0.015386
1986-03-17 -0.003025   0.0 -0.016248  0.013676
1986-03-18  0.034175   0.0 -0.007077 -0.004447
1986-03-19 -0.012128   0.0  0.011881 -0.004496
1986-03-20  0.063521   0.0  0.014024  0.007540

Shape: (10181, 4)


In [13]:
summary_stats = pd.DataFrame({
    "Mean_Daily_Return": daily_returns.mean(),
    "Daily_Volatility": daily_returns.std(),
    "Min_Daily_Return": daily_returns.min(),
    "Max_Daily_Return": daily_returns.max()
})

print(summary_stats)

      Mean_Daily_Return  Daily_Volatility  Min_Daily_Return  Max_Daily_Return
AAPL           0.001160          0.026675         -0.518475          0.332531
MSFT           0.001112          0.021993         -0.300027          0.195549
JPM            0.000657          0.022641         -0.276808          0.251152
XOM            0.000492          0.015944         -0.234023          0.178346


In [14]:
for ticker in tickers:
    print(f"\n{ticker} largest losses:")
    print(daily_returns[ticker].nsmallest(3))

    print(f"\n{ticker} largest gains:")
    print(daily_returns[ticker].nlargest(3))


AAPL largest losses:
Date
2000-09-29   -0.518475
1987-10-19   -0.243767
1993-07-16   -0.230778
Name: AAPL, dtype: float64

AAPL largest gains:
Date
1997-08-06    0.332531
1996-07-18    0.237243
1998-01-02    0.237219
Name: AAPL, dtype: float64

MSFT largest losses:
Date
1987-10-19   -0.300027
1987-10-26   -0.200303
2000-04-24   -0.155988
Name: MSFT, dtype: float64

MSFT largest gains:
Date
2000-10-19    0.195549
2008-10-13    0.186231
1987-10-21    0.166971
Name: MSFT, dtype: float64

JPM largest losses:
Date
1987-10-19   -0.276808
2009-01-20   -0.207415
2002-07-23   -0.181102
Name: JPM, dtype: float64

JPM largest gains:
Date
2009-01-21    0.251152
2009-03-23    0.246845
2009-03-10    0.226974
Name: JPM, dtype: float64

XOM largest losses:
Date
1987-10-19   -0.234023
2008-10-15   -0.139537
2020-03-09   -0.122257
Name: XOM, dtype: float64

XOM largest gains:
Date
1987-10-20    0.178346
2008-10-13    0.171940
2008-10-28    0.132672
Name: XOM, dtype: float64


## Stage 4 — Rolling Volatility and Regime Signal

We estimate changing market risk using rolling volatility.

A 20-day rolling volatility uses the most recent 20 daily returns.

We then create a market-wide stress measure by averaging rolling volatility across the four stocks.

In [15]:
rolling_vol_20 = daily_returns.rolling(window=20).std() * np.sqrt(252)

print(rolling_vol_20.head(25))

                AAPL      MSFT       JPM       XOM
Date                                              
1986-03-14       NaN       NaN       NaN       NaN
1986-03-17       NaN       NaN       NaN       NaN
1986-03-18       NaN       NaN       NaN       NaN
1986-03-19       NaN       NaN       NaN       NaN
1986-03-20       NaN       NaN       NaN       NaN
1986-03-21       NaN       NaN       NaN       NaN
1986-03-24       NaN       NaN       NaN       NaN
1986-03-25       NaN       NaN       NaN       NaN
1986-03-26       NaN       NaN       NaN       NaN
1986-03-27       NaN       NaN       NaN       NaN
1986-03-31       NaN       NaN       NaN       NaN
1986-04-01       NaN       NaN       NaN       NaN
1986-04-02       NaN       NaN       NaN       NaN
1986-04-03       NaN       NaN       NaN       NaN
1986-04-04       NaN       NaN       NaN       NaN
1986-04-07       NaN       NaN       NaN       NaN
1986-04-08       NaN       NaN       NaN       NaN
1986-04-09       NaN       NaN 

In [16]:
market_volatility = rolling_vol_20.mean(axis=1)

print(market_volatility.head(25))

Date
1986-03-14         NaN
1986-03-17         NaN
1986-03-18         NaN
1986-03-19         NaN
1986-03-20         NaN
1986-03-21         NaN
1986-03-24         NaN
1986-03-25         NaN
1986-03-26         NaN
1986-03-27         NaN
1986-03-31         NaN
1986-04-01         NaN
1986-04-02         NaN
1986-04-03         NaN
1986-04-04         NaN
1986-04-07         NaN
1986-04-08         NaN
1986-04-09         NaN
1986-04-10         NaN
1986-04-11    0.477995
1986-04-14    0.462858
1986-04-15    0.465612
1986-04-16    0.464792
1986-04-17    0.467764
1986-04-18    0.454234
dtype: float64


In [17]:
low_threshold = market_volatility.quantile(0.33)
high_threshold = market_volatility.quantile(0.67)

print("Low threshold:",
      low_threshold)

print("\nHigh threshold:",
      high_threshold)

Low threshold: 0.2334020088561999

High threshold: 0.3201699643611459


In [19]:
regime = pd.Series(
    index=market_volatility.index,
    dtype="object"
)

regime[market_volatility <= low_threshold] = "Low"
regime[
    (market_volatility > low_threshold)
    & (market_volatility < high_threshold)
] = "Normal"
regime[market_volatility >= high_threshold] = "High"

print(regime.value_counts())

Normal    3454
High      3354
Low       3354
Name: count, dtype: int64


## Stage 5 — Compare Stock Behavior Across Regimes

### 5.1 Average Returns by Regime

### 5.2 Volatility by Regime

### 5.3 Drawdowns by Regime

### 5.4 Correlation by Regime

We compare how strongly the stocks move together during Low, Normal, and High volatility regimes.

If correlations rise during stressed periods, diversification may become less effective exactly when it is needed most.

We want to answer:

How do AAPL, MSFT, JPM, and XOM behave differently when volatility conditions are Low, Normal, or High?

In [23]:
returns_with_regime = daily_returns.copy()

returns_with_regime["Regime"] = regime

print(returns_with_regime.head(25))

                AAPL      MSFT       JPM       XOM Regime
Date                                                     
1986-03-14  0.055077  0.000000  0.023762  0.015386    NaN
1986-03-17 -0.003025  0.000000 -0.016248  0.013676    NaN
1986-03-18  0.034175  0.000000 -0.007077 -0.004447    NaN
1986-03-19 -0.012128  0.000000  0.011881 -0.004496    NaN
1986-03-20  0.063521  0.000000  0.014024  0.007540    NaN
1986-03-21 -0.022707 -0.108030  0.002301 -0.013516    NaN
1986-03-24 -0.031866  0.000000  0.012714  0.001477    NaN
1986-03-25  0.044820  0.000000  0.002313  0.003109    NaN
1986-03-26  0.011576  0.000000  0.026122  0.022658    NaN
1986-03-27  0.000000  0.121114  0.011062 -0.002916    NaN
1986-03-31  0.000000  0.000000 -0.010941 -0.008965    NaN
1986-04-01 -0.034246 -0.108030 -0.042116 -0.004447    NaN
1986-04-02  0.000000  0.121114 -0.016147  0.013554    NaN
1986-04-03 -0.008669  0.000000  0.000000 -0.005945    NaN
1986-04-04 -0.011732  0.000000 -0.014170 -0.011935    NaN
1986-04-07  0.

In [21]:
regime_mean_returns = returns_with_regime.groupby("Regime")[
    ["AAPL", "MSFT", "JPM", "XOM"]
]. mean()

print("Average daily return by regime:",
      regime_mean_returns)

Average daily return by regime:             AAPL      MSFT       JPM       XOM
Regime                                        
High    0.001455  0.001512  0.000806  0.000546
Low     0.001476  0.000837  0.000738  0.000389
Normal  0.000543  0.000986  0.000420  0.000535


In [22]:
regime_volatility = returns_with_regime.groupby("Regime")[
    ["AAPL", "MSFT", "JPM", "XOM"]
].std()

print("\nDaily volatility by regime:",
      regime_volatility)


Daily volatility by regime:             AAPL      MSFT       JPM       XOM
Regime                                        
High    0.037109  0.030485  0.033528  0.021590
Low     0.015782  0.011835  0.011971  0.011131
Normal  0.022673  0.019089  0.016690  0.013265


In [24]:
# Cumulative growth of $1 for each stock
cummulative_growth = (1 + daily_returns).cumprod()

# Running peak for each stock
running_peak = cummulative_growth.cummax()

# Drawdown
drawdowns = cummulative_growth / running_peak - 1

print(drawdowns.head())

                AAPL  MSFT       JPM       XOM
Date                                          
1986-03-14  0.000000   0.0  0.000000  0.000000
1986-03-17 -0.003025   0.0 -0.016248  0.000000
1986-03-18  0.000000   0.0 -0.023210 -0.004447
1986-03-19 -0.012128   0.0 -0.011605 -0.008923
1986-03-20  0.000000   0.0  0.000000 -0.001450


In [25]:
drawdowns_with_regime = drawdowns.copy()

drawdowns_with_regime["Regime"] = regime

In [26]:
average_drawdown_by_regime = (
    drawdowns_with_regime
    .groupby("Regime")
    [["AAPL", "MSFT", "JPM", "XOM"]]
    .mean()
)

print("Average drawdown by regime:",
      average_drawdown_by_regime)

Average drawdown by regime:             AAPL      MSFT       JPM       XOM
Regime                                        
High   -0.363013 -0.255241 -0.286997 -0.123483
Low    -0.131450 -0.217366 -0.155616 -0.102648
Normal -0.310874 -0.215710 -0.202709 -0.102745


In [27]:
worst_drawdown_by_regime = (
    drawdowns_with_regime
    .groupby("Regime")
    [["AAPL", "MSFT", "JPM", "XOM"]]
    .min()
)

print("\nWorst drawdown by regime:",
      worst_drawdown_by_regime)


Worst drawdown by regime:             AAPL      MSFT       JPM       XOM
Regime                                        
High   -0.819694 -0.716511 -0.758338 -0.621136
Low    -0.753090 -0.594564 -0.479637 -0.381095
Normal -0.822429 -0.617383 -0.720695 -0.589012


### Drawdown Interpretation

High-volatility regimes generally coincide with deeper average drawdowns,
particularly for AAPL and JPM.

However, the worst drawdown does not always occur during the High-volatility
regime. Drawdown is path-dependent: a stock can remain far below its previous
peak even after volatility has declined from High to Normal.

Therefore, volatility regimes describe the intensity of recent price movements,
while drawdown measures how far an asset remains below its historical peak.

In [28]:
regime_correlations = {}

for regime_name in ["Low", "Normal", "High"]:
    regime_returns = returns_with_regime[
        returns_with_regime["Regime"] == regime_name
    ][["AAPL", "MSFT", "JPM", "XOM"]]

    regime_correlations[regime_name] = regime_returns.corr()

    print(f"\n{regime_name} regime correlation:",
          regime_correlations[regime_name])


Low regime correlation:           AAPL      MSFT       JPM       XOM
AAPL  1.000000  0.321723  0.238372  0.177520
MSFT  0.321723  1.000000  0.302937  0.164557
JPM   0.238372  0.302937  1.000000  0.356622
XOM   0.177520  0.164557  0.356622  1.000000

Normal regime correlation:           AAPL      MSFT       JPM       XOM
AAPL  1.000000  0.362920  0.230501  0.156316
MSFT  0.362920  1.000000  0.279593  0.194180
JPM   0.230501  0.279593  1.000000  0.295697
XOM   0.156316  0.194180  0.295697  1.000000

High regime correlation:           AAPL      MSFT       JPM       XOM
AAPL  1.000000  0.450369  0.335381  0.271720
MSFT  0.450369  1.000000  0.414082  0.370036
JPM   0.335381  0.414082  1.000000  0.417817
XOM   0.271720  0.370036  0.417817  1.000000


### Correlation Interpretation

Correlations generally increase during High-volatility regimes.

For example, AAPL-MSFT correlation rises from approximately 0.32 in the
Low-volatility regime to 0.45 in the High-volatility regime. Similar increases
appear across the other stock pairs.

This suggests that diversification benefits may weaken during periods of market
stress because assets become more synchronized when volatility is elevated.

The relationship is not perfectly monotonic across Low, Normal, and High
regimes for every pair, but the High-volatility regime produces the strongest
correlation for all six pairwise relationships in this sample.

## Stage 6 — Trading Volume and Stress Regimes

This section examines whether trading activity changes across Low, Normal, and High volatility regimes.

We compare average trading volume by regime to see whether stressed market conditions are associated with increased trading activity.

In [29]:
volume_with_regime = volume_data.copy()

volume_with_regime["Regime"] = regime

print(volume_with_regime.head(25))

                  AAPL          MSFT           JPM           XOM Regime
Date                                                                   
1986-03-13   138223764  1.496329e+09  1.162157e+06  1.797470e+07    NaN
1986-03-14   458725551  4.469023e+08  1.160078e+06  2.063858e+07    NaN
1986-03-17   141507822  1.931286e+08  7.235918e+05  1.474184e+07    NaN
1986-03-18   297219767  9.827675e+07  5.650572e+05  1.750917e+07    NaN
1986-03-19   226332410  6.945781e+07  3.191425e+05  1.067231e+07    NaN
1986-03-20  1077675436  8.474437e+07  7.356576e+05  9.214942e+06    NaN
1986-03-21   310356004  8.699977e+07  7.385728e+05  2.237139e+07    NaN
1986-03-24   350805852  9.468483e+07  1.088921e+06  1.306075e+07    NaN
1986-03-25   335026425  4.652796e+07  5.267771e+05  7.497651e+06    NaN
1986-03-26   264779766  3.299559e+07  9.865600e+05  9.604179e+06    NaN
1986-03-27   261041881  2.443344e+07  7.556283e+05  1.642293e+07    NaN
1986-03-31   223849348  1.866966e+07  8.633995e+05  9.514950e+06

In [30]:
average_volume_by_regime = (
    volume_with_regime
    .groupby("Regime")[["AAPL", "MSFT", "JPM", "XOM"]]
    .mean()
)

print("Average trading volume by regime:",
      average_volume_by_regime)

Average trading volume by regime:                 AAPL          MSFT           JPM           XOM
Regime                                                        
High    4.789146e+08  9.629030e+07  2.019988e+07  2.141994e+07
Low     3.492465e+08  4.887337e+07  1.875698e+07  2.046780e+07
Normal  3.339929e+08  7.690918e+07  1.378601e+07  1.710000e+07


In [31]:
median_volume_by_regime = (
    volume_with_regime
    .groupby("Regime")[["AAPL", "MSFT", "JPM", "XOM"]]
    .median()
)

print("\nMedian trading volume by regime:",
      median_volume_by_regime)


Median trading volume by regime:                AAPL          MSFT           JPM           XOM
Regime                                                       
High    323064970.0  8.662358e+07  9.809250e+06  1.547821e+07
Low     203984206.5  3.809672e+07  1.442193e+07  1.782240e+07
Normal  213329705.0  6.791013e+07  8.628445e+06  1.341656e+07


### Volume Interpretation

Trading activity generally increases during High-volatility regimes, but the pattern differs across stocks.

AAPL and MSFT show both higher average and higher median trading volume during High-volatility periods, suggesting that elevated trading activity is broadly persistent during stressed conditions.

JPM and XOM show higher average volume during High-volatility regimes, but their median volume is higher during Low-volatility periods. This indicates that a smaller number of extreme high-volume days may be increasing the High-regime average.

Overall, the results suggest that market stress is associated with greater trading activity, although the strength and consistency of this relationship vary across assets.

## Stage 7 — Stress-Period Case Studies

We examine selected historical stress periods and compare:

- average return
- volatility
- average drawdown
- dominant volatility regime

The goal is to see whether the regime framework captures periods of severe market stress.

In [35]:
stress_periods = {
    "1987 Crash": ("1987-09-01", "1987-11-30"),
    "Financial Crisis": ("2008-09-01", "2009-03-31"),
    "COVID Crash": ("2020-02-01", "2020-05-31")
}

In [39]:
start_date, end_date = stress_periods["1987 Crash"]

returns_1987 = regime.loc[start_date:end_date]

print(returns_1987.head())
print("\nShape:", returns_1987.shape)

Date
1987-09-01      High
1987-09-02      High
1987-09-03      High
1987-09-04      High
1987-09-08    Normal
dtype: object

Shape: (63,)


In [37]:
mean_return_1987 = returns_1987.mean()

volatility_1987 = returns_1987.std()

drawdown_1987 = drawdowns.loc[start_date:end_date]

average_drawdown_1987 = drawdown_1987.mean()
worst_drawdown_1987 = drawdown_1987.min()

regime_1987 = regime.loc[start_date:end_date]
regime_counts_1987 = returns_1987.value_counts()

In [40]:
print("1987 Average Daily Return:",
      mean_return_1987)

print("\n1987 Daily Volatility:",
      volatility_1987)

print("\n1987 Average Drawdown:",
      average_drawdown_1987)

print("\n1987 Worst Drawdown:",
      worst_drawdown_1987)

print("\n1987 Regime Count:",
      regime_1987.value_counts())

1987 Average Daily Return: AAPL   -0.005708
MSFT   -0.002022
JPM    -0.005794
XOM    -0.003646
dtype: float64

1987 Daily Volatility: AAPL    0.063576
MSFT    0.066928
JPM     0.042796
XOM     0.047471
dtype: float64

1987 Average Drawdown: AAPL   -0.211083
MSFT   -0.222918
JPM    -0.350519
XOM    -0.111526
dtype: float64

1987 Worst Drawdown: AAPL   -0.527212
MSFT   -0.510464
JPM    -0.500551
XOM    -0.331694
dtype: float64

1987 Regime Count: High      53
Normal    10
Name: count, dtype: int64


### 1987 Case Study Interpretation

The 1987 stress period was dominated by the High-volatility regime, with 53 of 63 trading days classified as High.

All four stocks experienced negative average daily returns during the period, while daily volatility increased sharply.

Drawdowns were also severe, with AAPL, MSFT, and JPM each experiencing worst drawdowns of approximately 50% or more during the selected window.

This case study suggests that the regime framework successfully identifies the 1987 period as a prolonged episode of elevated market stress.

In [50]:
start_date, end_date = stress_periods["Financial Crisis"]

returns_2008 = daily_returns.loc[start_date:end_date]

print(returns_2008.head())
print("\nShape:", returns_2008.shape)

                AAPL      MSFT       JPM       XOM
Date                                              
2008-09-02 -0.019626 -0.006985  0.012992 -0.033695
2008-09-03  0.004593 -0.007227  0.018631  0.009109
2008-09-04 -0.034326 -0.020391 -0.045361 -0.024036
2008-09-05 -0.006463 -0.026652  0.044499 -0.006895
2008-09-08 -0.014104  0.018134  0.049345  0.015134

Shape: (146, 4)


In [51]:
# Numeric return calculations
mean_return_2008 = returns_2008.mean()
volatility_2008 = returns_2008.std()

# Drawdown calculations
drawdown_2008 = drawdowns.loc[start_date:end_date]

average_drawdown_2008 = drawdown_2008.mean()
worst_drawdown_2008 = drawdown_2008.min()

# Regime labels
regime_2008 = regime.loc[start_date:end_date]
regime_counts_2008 = regime_2008.value_counts()

In [52]:
print("2008-2009 Average Daily Return:",
      mean_return_2008)

print("\n2008-2009 Daily Volatility:",
      volatility_2008)

print("\n2008-2009 Average Drawdown:",
      average_drawdown_2008)

print("\n2008-2009 Worst Drawdown:",
      worst_drawdown_2008)

print("\n2008-2009 Regime Counts:",
      regime_counts_2008)

2008-2009 Average Daily Return: AAPL   -0.002326
MSFT   -0.001887
JPM     0.000600
XOM    -0.000243
dtype: float64

2008-2009 Daily Volatility: AAPL    0.043400
MSFT    0.040929
JPM     0.080057
XOM     0.041836
dtype: float64

2008-2009 Average Drawdown: AAPL   -0.494420
MSFT   -0.614751
JPM    -0.510432
XOM    -0.217124
dtype: float64

2008-2009 Worst Drawdown: AAPL   -0.608644
MSFT   -0.716511
JPM    -0.751370
XOM    -0.345439
dtype: float64

2008-2009 Regime Counts: High    146
Name: count, dtype: int64


In [53]:
start_date, end_date = stress_periods["COVID Crash"]

returns_2020 = daily_returns.loc[start_date:end_date]
regime_2020 = regime.loc[start_date:end_date]

print(returns_2020.head())
print("\nShape:", returns_2020.shape)

                AAPL      MSFT       JPM       XOM
Date                                              
2020-02-03 -0.002776  0.024401  0.007841 -0.022407
2020-02-04  0.033008  0.032914  0.014351 -0.012520
2020-02-05  0.008107 -0.001248  0.016763  0.046055
2020-02-06  0.011739  0.020666  0.000150 -0.013562
2020-02-07 -0.013592  0.001441 -0.002944 -0.006648

Shape: (82, 4)


In [54]:
mean_return_2020 = returns_2020.mean()
volatility_2020 = returns_2020.std()

drawdown_2020 = drawdowns.loc[start_date:end_date]

average_drawdown_2020 = drawdown_2020.mean()
worst_drawdown_2020 = drawdown_2020.min()

regime_counts_2020 = regime_2020.value_counts()

In [55]:
print("2020 Average Daily Return:",
      mean_return_2020)

print("\n2020 Daily Volatility:",
      volatility_2020)

print("\n2020 Average Drawdown:",
      average_drawdown_2020)

print("\n2020 Worst Drawdown:",
      worst_drawdown_2020)

print("\n2020 Regime Counts:",
      regime_counts_2020)

2020 Average Daily Return: AAPL    0.001140
MSFT    0.001787
JPM    -0.002334
XOM    -0.002396
dtype: float64

2020 Daily Volatility: AAPL    0.038961
MSFT    0.040730
JPM     0.051051
XOM     0.044749
dtype: float64

2020 Average Drawdown: AAPL   -0.118944
MSFT   -0.093554
JPM    -0.263276
XOM    -0.444110
dtype: float64

2020 Worst Drawdown: AAPL   -0.314261
MSFT   -0.280435
JPM    -0.436321
XOM    -0.621136
dtype: float64

2020 Regime Counts: High      65
Normal    15
Low        2
Name: count, dtype: int64


### 2020 COVID Crash Interpretation

The COVID stress window was dominated by the High-volatility regime, with 65 of 82 trading days classified as High.

Volatility increased across all four stocks, with JPM showing the highest daily volatility at approximately 5.1%.

The effects of the crisis were uneven across assets. AAPL and MSFT produced positive average daily returns over the full window, while JPM and XOM produced negative average daily returns.

XOM experienced the deepest drawdown at approximately -62%, followed by JPM at approximately -44%.

Overall, the regime framework identifies the COVID period as a major stress episode, while also showing that individual stocks responded differently depending on their sector and recovery behavior.

## Stage 8 — Final Research Summary

This section summarizes the main findings from the regime analysis.

The goal is to combine the most important results into a compact research-style conclusion covering:

- returns across regimes
- volatility across regimes
- drawdowns across regimes
- correlation behavior
- trading volume
- major historical stress periods

In [56]:
regime_summary = pd.concat(
    {
        "Mean Return": regime_mean_returns,
        "Volatility": regime_volatility
    },
    axis=1
)

print(regime_summary)

       Mean Return                               Volatility            \
              AAPL      MSFT       JPM       XOM       AAPL      MSFT   
Regime                                                                  
High      0.001455  0.001512  0.000806  0.000546   0.037109  0.030485   
Low       0.001476  0.000837  0.000738  0.000389   0.015782  0.011835   
Normal    0.000543  0.000986  0.000420  0.000535   0.022673  0.019089   

                            
             JPM       XOM  
Regime                      
High    0.033528  0.021590  
Low     0.011971  0.011131  
Normal  0.016690  0.013265  


In [57]:
drawdown_summary = pd.concat(
    {
        "Average Drawdown": average_drawdown_by_regime,
        "Worst Drawdown": worst_drawdown_by_regime
    },
    axis=1
)

print(drawdown_summary)

       Average Drawdown                               Worst Drawdown  \
                   AAPL      MSFT       JPM       XOM           AAPL   
Regime                                                                 
High          -0.363013 -0.255241 -0.286997 -0.123483      -0.819694   
Low           -0.131450 -0.217366 -0.155616 -0.102648      -0.753090   
Normal        -0.310874 -0.215710 -0.202709 -0.102745      -0.822429   

                                      
            MSFT       JPM       XOM  
Regime                                
High   -0.716511 -0.758338 -0.621136  
Low    -0.594564 -0.479637 -0.381095  
Normal -0.617383 -0.720695 -0.589012  


In [58]:
stress_regime_summary = pd.DataFrame(
    {
        "1987 Crash": regime_1987.value_counts(),
        "2008-2009 Crisis": regime_2008.value_counts(),
        "2020 COVID": regime_2020.value_counts()
    }
).fillna(0)

print(stress_regime_summary)

        1987 Crash  2008-2009 Crisis  2020 COVID
High          53.0             146.0          65
Low            0.0               0.0           2
Normal        10.0               0.0          15


### Final Research Findings

The regime analysis reveals several consistent patterns across AAPL, MSFT, JPM, and XOM.

First, volatility is substantially higher during High-volatility regimes for all four stocks. This confirms that the regime classification successfully separates calmer periods from stressed market conditions.

Second, drawdowns tend to be deeper during High-volatility regimes. Although the single worst drawdown does not always occur during the High regime, average drawdowns are generally more severe when volatility is elevated. This reflects the path-dependent nature of drawdowns: an asset may remain far below its previous peak even after volatility begins to decline.

Third, correlations increase during High-volatility regimes. All six pairwise stock correlations are stronger in the High regime than in the Low regime. This suggests that diversification benefits may weaken during periods of market stress because assets become more synchronized.

Fourth, trading activity generally increases during High-volatility periods. Average volume is highest in the High regime for all four stocks, although median volume shows a more mixed pattern for JPM and XOM.

Finally, the regime framework identifies major historical stress periods effectively. The 1987 crash, the 2008–2009 Financial Crisis, and the 2020 COVID crash were all dominated by High-volatility classifications.

Overall, the results suggest that market stress is characterized not only by higher volatility, but also by deeper drawdowns, stronger cross-asset correlations, and elevated trading activity.

In [59]:
regime_summary.to_csv(results_folder / "regime_summary_csv")

drawdown_summary.to_csv(results_folder / "drawdown_summary.csv")

average_volume_by_regime.to_csv(
    results_folder / "average_volume_by_regime.csv"
)

median_volume_by_regime.to_csv(
    results_folder / "median_volume_by_regime.csv"
)

stress_regime_summary.to_csv(
    results_folder / "stress_regime_summary.csv"
)

print("Results saved successfully")

Results saved successfully
